In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .appName("ETL-WordPress-to-HDFS") \
    .master("local[*]") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

# 1. Extract from WordPress PostgreSQL
df = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/wordpress") \
    .option("dbtable", "sales_data") \
    .option("user", "hiveuser") \
    .option("password", "hivepassword") \
    .option("driver", "org.postgresql.Driver") \
    .load()

print(f"Extracted {df.count()} records from WordPress")
df.show()

# 2. Clean
df = df.dropDuplicates().dropna(subset=["product", "region"])

# 3. Transform
df = df.withColumn("chiffre_affaires", F.col("quantity") * F.col("price"))

# 4. Load to HDFS
df.write.mode("overwrite").csv("hdfs://namenode:9000/data/wordpress_sales/raw", header=True)
print("Saved to HDFS!")

# 5. Save to Hive Metastore
df.write.mode("overwrite").saveAsTable("wordpress_sales")
print("Saved to Metastore!")

spark.sql("SELECT * FROM wordpress_sales").show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 18:03:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/16 18:03:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Extracted 3 records from WordPress


+---+-----+-----------------+-----------------+--------+--------+------+--------------------+
| id| name|            email|          product|quantity|   price|region|          created_at|
+---+-----+-----------------+-----------------+--------+--------+------+--------------------+
|  1|  idk|      idk@idk.com|                 |       2| 5454.00|      |2026-05-16 17:49:...|
|  2|  idk|     idk@idk.com7|Clavier Mecanique|    NULL| 6554.00|      |2026-05-16 17:52:...|
|  3|urmom|urmom@myhouse.com|Clavier Mecanique|    NULL|64545.00|      |2026-05-16 18:01:...|
+---+-----+-----------------+-----------------+--------+--------+------+--------------------+



Saved to HDFS!


26/05/16 18:03:47 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/16 18:03:47 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/16 18:03:50 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/16 18:03:50 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.18.0.8
26/05/16 18:03:52 WARN HadoopFSUtils: The directory file:/opt/spark/work-dir/spark-warehouse/wordpress_sales was not found. Was it deleted very recently?
26/05/16 18:03:52 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/05/16 18:03:59 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/05/16 18:03:59 WARN HiveConf: HiveConf of name hive.internal.ss

Saved to Metastore!


[Stage 10:>                                                         (0 + 1) / 1]

+---+-----+-----------------+-----------------+--------+--------+------+--------------------+----------------+
| id| name|            email|          product|quantity|   price|region|          created_at|chiffre_affaires|
+---+-----+-----------------+-----------------+--------+--------+------+--------------------+----------------+
|  1|  idk|      idk@idk.com|                 |       2| 5454.00|      |2026-05-16 17:49:...|        10908.00|
|  2|  idk|     idk@idk.com7|Clavier Mecanique|    NULL| 6554.00|      |2026-05-16 17:52:...|            NULL|
|  3|urmom|urmom@myhouse.com|Clavier Mecanique|    NULL|64545.00|      |2026-05-16 18:01:...|            NULL|
+---+-----+-----------------+-----------------+--------+--------+------+--------------------+----------------+

